In [ ]:
from huggingface_hub import InferenceClient
import os

In [ ]:
# Import de la clef d'API (via variable d'environnement) et du modèle
api_key = os.getenv("HF_API_KEY")
modelID="meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
class Discussion ():
    def __init__(self, modelID, api_key, window=5, max_token=100, temp=0.2):
        self.client = InferenceClient(api_key=api_key)
        self.modelID = modelID
        self.window = window #longueur de la fenêtre glissante de mémoire
        self.max_token = max_token #longueur de la réponse de l'IA
        self.temp = temp
        self.hist = []
        self.intent = None
        
        #Définition des prompts système en fonction du type de discussion
        self.prompt_sys = {
            "INTENT" : ("Tu es un classificateur d'intentions. Analyse le message de l'utilisateur et réponds par UN SEUL MOT : "
            "'MEDICAL' si le message concerne des symptômes, une maladie ou une question de santé ; "
            "'ADMIN' si le message concerne un rendez-vous de téléconsultation, un remboursement ou le fonctionnement de Tessan ; "
            "'AUTRE' pour tout le reste."),
            "MEDICAL" : "Tu es un chatbot médical, répond de façon courte et concise",
            "ADMIN" : "Tu répond aux questions administratives",
            "AUTRE" : "Je ne peux pas vous aider"
        }

    def getlog(self):
        return self.hist
    
    def callAI(self, context):
        answer_gen = self.client.chat.completions.create(
                model=self.modelID,
                messages=context,
                max_tokens=self.max_token,
                temperature=self.temp)
        return answer_gen.choices[0].message.content
    
    def process_input(self, user_input):
        self.hist.append({"role":"user","content":user_input}) #Ajout de l'entrée de l'utilisateur à l'historique
        
        #Test de l'intention si elle n'a pas encore été détectée
        if self.intent is None : 
            print("Détection de l'intention")
            context_detection = [{"role":"system","content":self.prompt_sys["INTENT"]}] + self.hist[-1:]
            intent = self.callAI(context_detection).upper()

            if "MEDICAL" in intent: self.intent = "MEDICAL"
            elif "ADMIN" in intent: self.intent = "ADMIN"
            else: self.intent = "AUTRE"
            print(self.intent)
        
        #Réponse à la requete en fonction de l'intention
        if self.intent == "AUTRE":
            answer_fin= self.prompt_sys["AUTRE"]
        else :
            #Créer la fenêtre de contexte :
            context = [{"role":"system","content":self.prompt_sys[self.intent]}] + self.hist[-self.window:]
            
            #Call l'IA pour répondre :
            answer_fin = self.callAI(context)
            if self.intent == "MEDICAL" :
                answer_fin += "\n\nSouhaitez-vous passer en téléconsultation maintenant ?"
            
        self.hist.append({"role":"assistant","content":answer_fin})

        return answer_fin


In [ ]:
chat = Discussion(modelID=modelID,api_key=api_key)

In [ ]:
user_input=input()

In [ ]:
print(chat.process_input(user_input=user_input))

In [ ]:
user_input = input()

In [ ]:
print(chat.process_input(user_input=user_input))